# Tarea 7: RAG y Agente Inteligente de Mantenimiento
Este cuaderno unifica los puntos solicitados en la tarea (RAG Industrial, RAG Curso y Agente de Mantenimiento).

**Nota:** Asegúrate de tener el servidor local de **LM Studio** encendido (puerto 1234) corriendo el modelo `Gemma 2 2B`.

## 1. Configuración Inicial e Importaciones

In [1]:
import os
import json
from datetime import datetime

# LangChain: Componentes principales
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

# LangGraph para el agente (método moderno)
from langgraph.prebuilt import create_react_agent

# Inicializamos el modelo de LM Studio y los Embeddings
print("Inicializando modelo Gemma 2 2B en LM Studio...")
llm = ChatOpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio", temperature=0)

print("Inicializando modelo de Embeddings (all-MiniLM-L6-v2)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("¡Listo!")

C:\Users\kevin\AppData\Local\Temp\ipykernel_30244\450627595.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Inicializando modelo Gemma 2 2B en LM Studio...
Inicializando modelo de Embeddings (all-MiniLM-L6-v2)...


C:\Users\kevin\AppData\Local\Temp\ipykernel_30244\450627595.py:25: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
c:\Users\kevin\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3186.48it/s]


¡Listo!


## 2. Función Universal para Crear RAGs
Creamos una función genérica basada en LCEL que podemos reutilizar tanto para los manuales como para los documentos del curso.

In [2]:
def setup_rag(docs_dir, persist_dir):
    if not os.path.exists(docs_dir):
        print(f"[!] El directorio '{docs_dir}' no existe. Asegúrate de crearlo y agregar archivos .txt.")
        return None, None
        
    # Cargar documentos (usando utf-8 por si estás en Windows)
    loader = DirectoryLoader(docs_dir, glob="**/*.txt", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    docs = loader.load()
    if not docs:
        print(f"[!] No hay archivos .txt en {docs_dir}.")
        return None, None
        
    # Dividir texto en trozos
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    texts = text_splitter.split_documents(docs)
    
    # Crear Vector DB
    vectordb = Chroma.from_documents(documents=texts, embedding=embeddings, persist_directory=persist_dir)
    retriever = vectordb.as_retriever(search_kwargs={"k": 3})
    
    # Crear Prompt LCEL
    template = """Usa el siguiente contexto para responder a la pregunta. Si no sabes la respuesta, simplemente di que no lo sabes.\n\nContexto: {context}\n\nPregunta: {question}\nRespuesta útil:"""
    prompt = PromptTemplate.from_template(template)
    
    def format_docs(docs_list):
        return "\n\n".join(doc.page_content for doc in docs_list)
        
    # Cadena LCEL
    qa_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()} 
        | prompt 
        | llm 
        | StrOutputParser()
    )
    print(f"[+] RAG configurado exitosamente para: {docs_dir}")
    return qa_chain, retriever

## 3. Prueba RAG Industrial (Punto 2)

In [3]:
qa_industrial, retriever_ind = setup_rag("manuales_maquinas", "chroma_db_industrial")

if qa_industrial:
    pregunta_ind = "¿Qué repuesto necesito si la bomba B-101 tiene una fuga en el sello mecánico?"
    print("\n--- PREGUNTA ---")
    print(pregunta_ind)
    print("\n--- RESPUESTA ---")
    print(qa_industrial.invoke(pregunta_ind))

[+] RAG configurado exitosamente para: manuales_maquinas

--- PREGUNTA ---
¿Qué repuesto necesito si la bomba B-101 tiene una fuga en el sello mecánico?

--- RESPUESTA ---
REP-001 (Kit de sellos y rodamiento).


## 4. Prueba RAG del Curso (Punto 3)

In [4]:
qa_curso, retriever_curso = setup_rag("documentos_curso", "chroma_db_curso")

if qa_curso:
    pregunta_curso = "¿Qué es RAG y cuáles son sus componentes según la Unidad 2?"
    print("\n--- PREGUNTA ---")
    print(pregunta_curso)
    print("\n--- RESPUESTA ---")
    print(qa_curso.invoke(pregunta_curso))

[+] RAG configurado exitosamente para: documentos_curso

--- PREGUNTA ---
¿Qué es RAG y cuáles son sus componentes según la Unidad 2?

--- RESPUESTA ---
RAG (Retrieval-Augmented Generation) es una técnica que permite a un LLM (Large Language Model) acceder a una base de datos externa de documentos para responder preguntas con mayor precisión y reducir las alucinaciones.

Los componentes del RAG son:
1. **Base de datos vectorial (Vector DB):** Como ChromaDB o Pinecone.
2. **Embeddings:** Representaciones matemáticas de texto.
3. **LLM:** El modelo que genera la respuesta, como Llama 3 o GPT-4.


## 5. Agente de Mantenimiento (Punto 4)
Definimos las herramientas y el Agente inteligente usando LangGraph.

In [5]:
# Bases de datos simuladas
INVENTARIO_REPUESTOS = {
    "REP-001": {"nombre": "Rodamiento de bolas SKF", "ubicacion": "Estante A1", "cantidad": 5},
    "REP-002": {"nombre": "Filtro de aceite", "ubicacion": "Estante B3", "cantidad": 0},
}
HISTORIAL_FALLAS = []
ORDENES_TRABAJO = []

# Creación de las Tools
@tool
def buscar_repuesto(codigo: str) -> str:
    """Busca información de un repuesto utilizando su código (ej. REP-001)."""
    if codigo in INVENTARIO_REPUESTOS:
        return f"Encontrado: {INVENTARIO_REPUESTOS[codigo]['nombre']}. Ubicación: {INVENTARIO_REPUESTOS[codigo]['ubicacion']}."
    return f"Código {codigo} no encontrado."

@tool
def consultar_almacen(codigo: str) -> str:
    """Consulta la cantidad disponible de un repuesto en el almacén por su código."""
    if codigo in INVENTARIO_REPUESTOS:
        cantidad = INVENTARIO_REPUESTOS[codigo]['cantidad']
        return f"El repuesto {codigo} tiene {cantidad} unidades en almacén."
    return f"Código {codigo} no encontrado."

@tool
def generar_orden_trabajo(falla: str, equipo: str) -> str:
    """Genera una orden de trabajo indicando la falla y el equipo."""
    id_orden = f"OT-{len(ORDENES_TRABAJO) + 1:03d}"
    ORDENES_TRABAJO.append({"id": id_orden, "equipo": equipo, "falla": falla, "estado": "Pendiente"})
    return f"Orden {id_orden} generada exitosamente."

@tool
def actualizar_historial_fallas(falla: str, equipo: str, notas: str = "") -> str:
    """Actualiza el historial de fallas de un equipo."""
    HISTORIAL_FALLAS.append({"equipo": equipo, "falla": falla, "notas": notas})
    return f"Historial actualizado para {equipo}."

tools = [buscar_repuesto, consultar_almacen, generar_orden_trabajo, actualizar_historial_fallas]

# Configuración del Agente con LangGraph
agent_executor = create_react_agent(llm, tools)

C:\Users\kevin\AppData\Local\Temp\ipykernel_30244\4009089327.py:41: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


### Prueba del Agente de Mantenimiento

In [6]:
consulta_agente = "Tengo una falla severa en la Bomba B-101. Genera una orden de trabajo, actualiza el historial y luego verifica si tenemos en almacén el repuesto REP-001."

print("\n--- CONSULTA AL AGENTE ---")
print(consulta_agente)

print("\n--- EJECUCIÓN ---")
respuesta = agent_executor.invoke({
    "messages": [
        SystemMessage(content="Eres un excelente asistente de mantenimiento industrial. Usa las herramientas provistas para resolver el problema paso a paso. Responde en español."),
        HumanMessage(content=consulta_agente)
    ]
})

print("\n--- RESPUESTA FINAL ---")
print(respuesta["messages"][-1].content)


--- CONSULTA AL AGENTE ---
Tengo una falla severa en la Bomba B-101. Genera una orden de trabajo, actualiza el historial y luego verifica si tenemos en almacén el repuesto REP-001.

--- EJECUCIÓN ---

--- RESPUESTA FINAL ---
He completado todas las acciones solicitadas:

1.  **Generación de Orden de Trabajo:** Se ha generado la orden de trabajo para la **Bomba B-101** por la falla reportada.
    *   *Resultado:* Orden OT-001 generada exitosamente.

2.  **Actualización del Historial de Fallas:** Se ha actualizado el historial de fallas del equipo.
    *   *Resultado:* Historial actualizado para Bomba B-101.

3.  **Verificación de Repuesto en Almacén:** Se ha consultado la disponibilidad del repuesto **REP-001**.
    *   *Resultado:* El repuesto REP-001 tiene **5 unidades** en almacén.

La orden de trabajo ha sido generada, el historial actualizado y se confirma que hay 5 unidades del repuesto REP-001 disponibles en almacén para la reparación.
